# 03 · Execution Time Analysis

This notebook analyses execution time across languages and its relationship to energy.

**Units:** Time values are in **seconds (s)** (converted from raw µs at load time).
Energy values are in **Joules (J)**.

**Key questions:**
- Which languages execute fastest?
- How does time correlate with CPU and memory energy?

**Methodology:** Rankings (time) and the heatmap use the **two-step mean** (equal
benchmark weight) from `results_clean_runs.csv` via `lang_means()`. Spearman correlation is
used for the energy↔time relationship (more robust than Pearson for right-skewed data),
and the compiler comparison keeps the non-parametric Kruskal-Wallis / Mann-Whitney tests.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))   # make the shared style module importable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from itertools import combinations

import importlib
import plot_style as ps
importlib.reload(ps)   # pick up edits to plot_style.py without a kernel restart
ps.apply_style()

# Canonical constants — single source: plot_style.
COL_CPU_ENERGY, COL_MEM_ENERGY = ps.COL_CPU_ENERGY, ps.COL_MEM_ENERGY
COL_TIME                       = ps.COL_TIME
COL_CPU_CARBON, COL_MEM_CARBON = ps.COL_CPU_CARBON, ps.COL_MEM_CARBON
COMPILER        = ps.COMPILER
COMPILER_COLORS = ps.COMPILER_COLORS
COMPILER_ORDER  = ps.COMPILER_ORDER
MEANPROPS       = ps.MEANPROPS
ALPHA           = ps.ALPHA

OUTPUTS_DIR = Path('outputs'); OUTPUTS_DIR.mkdir(exist_ok=True)

# Single source of truth: per-run rows (df) + per-cell means with EDP (df_mean).
df      = ps.load_runs()
df_mean = ps.cell_means(df)

def lang_means(cols):
    """Per-language two-step mean (equal benchmark weight) for column(s) `cols`."""
    return ps.lang_means(df_mean, cols)

print(f"Runs: {df.shape} | Cell-means: {df_mean.shape} | "
      f"{df['language'].nunique()} languages \u00d7 {df['benchmark'].nunique()} benchmarks")
df_mean.head(3)

## 1. Execution Time Ranking

Languages ranked by **mean** execution time (s) — fastest at the top.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
time_rank = lang_means(COL_TIME).sort_values()   # order bars by the plotted metric
colors = [COMPILER_COLORS[COMPILER[l]] for l in time_rank.index]
bars = ax.barh(time_rank.index, time_rank.values, color=colors, alpha=0.85, edgecolor='white')

# value label at the end of each bar
x_max = time_rank.max()
for bar, val in zip(bars, time_rank.values):
    ax.text(bar.get_width() + x_max * 0.01, bar.get_y() + bar.get_height() / 2,
            f'{val:,.2f}', va='center', ha='left', fontsize=8, color='#333333')
ax.set_xlim(0, x_max * 1.15)

ax.set_title('', fontsize=12)
ax.set_xlabel('Mean Execution Time (s)')
ax.set_ylabel('')
ax.invert_yaxis()   # fastest (lowest) at the top
legend_handles = [mpatches.Patch(color=COMPILER_COLORS[p], label=p, alpha=0.85)
                  for p in COMPILER_ORDER]
ax.legend(handles=legend_handles, title='Execution Model', loc='upper right')
plt.tight_layout()
ps.save_fig(fig, '03_time_ranking')
plt.show()

> **Takeaway:** the time ranking closely mirrors the CPU-energy ranking — AOT native binaries are fastest, the interpreted languages slowest — confirming time and energy track together.

## 2. Correlation Summary — Spearman ρ with Rea–Parker effect size

Spearman rank correlation between the three primary metrics, on the per-language
means (N = 18). Rank-based, so robust to the right-skewed distributions.
Following **Rea & Parker**, the modulus of each ρ is given a nominal effect-size
label:

| \|ρ\| | Classification |
|------|----------------|
| 0.00 – 0.10 | Negligible |
| 0.10 – 0.20 | Weak |
| 0.20 – 0.40 | Moderate |
| 0.40 – 0.60 | Relatively strong |
| 0.60 – 0.80 | Strong |
| 0.80 – 1.00 | Very strong |

In [ ]:
# Spearman rank correlations between the three primary metrics, on the
# per-language two-step means (N = 18); rank-based, robust to the right-skew.
lm = df_mean.groupby('language')[[COL_CPU_ENERGY, COL_MEM_ENERGY, COL_TIME]].mean()

def classify(r):
    """Rea & Parker nominal effect-size label for |Spearman rho|."""
    a = abs(r)
    if a < 0.10: return 'Negligible'
    if a < 0.20: return 'Weak'
    if a < 0.40: return 'Moderate'
    if a < 0.60: return 'Relatively strong'
    if a < 0.80: return 'Strong'
    return 'Very strong'

pairs = [
    ('CPU Energy × Execution Time',    COL_CPU_ENERGY, COL_TIME),
    ('CPU Energy × Memory Energy',     COL_CPU_ENERGY, COL_MEM_ENERGY),
    ('Memory Energy × Execution Time', COL_MEM_ENERGY, COL_TIME),
]
rows = []
for name, a, b in pairs:
    rho, p = stats.spearmanr(lm[a], lm[b])
    rows.append({'Correlation': name, 'Spearman ρ': round(rho, 3),
                 'p-value': p, 'N': len(lm), 'Classification': classify(rho)})
corr_table = pd.DataFrame(rows).set_index('Correlation')
corr_table.to_csv(OUTPUTS_DIR / 'spearman_correlations.csv')
print('Saved → outputs/spearman_correlations.csv')

# Display-formatted copy (strings) for the styled table figure (PNG + PDF).
disp = pd.DataFrame({
    'Spearman ρ':     corr_table['Spearman ρ'].map(lambda v: f'{v:.3f}'),
    'p-value':        corr_table['p-value'].map(lambda v: f'{v:.1e}'),
    'N':              corr_table['N'].astype(str),
    'Classification': corr_table['Classification'],
}, index=corr_table.index)
ps.styled_table_fig(
    disp,
    'Spearman Correlations between Primary Metrics (per-language means, N=18)',
    '03_spearman_correlation_table',
    highlight_col='Classification',
)
corr_table

## 3. Execution Model Speed Comparison

Kruskal-Wallis test on execution time across compiler groups, followed by pairwise
Mann-Whitney U tests with Bonferroni correction.

In [ ]:
import matplotlib.ticker as mticker

# Violin distribution of execution time per compiler (log scale — time spans
# several orders of magnitude). Mirrors the energy violin in notebook 02.
fig, ax = plt.subplots(figsize=(8, 6))
groups = [np.log10(df[df['compiler'] == p][COL_TIME].values) for p in COMPILER_ORDER]
parts = ax.violinplot(groups, positions=range(len(COMPILER_ORDER)),
                      showmedians=True, showmeans=True)
for pc, p in zip(parts['bodies'], COMPILER_ORDER):
    pc.set_facecolor(COMPILER_COLORS[p])
    pc.set_alpha(0.85)
for key in ('cmedians', 'cmeans', 'cbars', 'cmins', 'cmaxes'):
    if key in parts:
        parts[key].set_edgecolor('#333333')
        parts[key].set_linewidth(1)

ax.set_xticks(range(len(COMPILER_ORDER)))
ax.set_xticklabels(COMPILER_ORDER)
# show log-spaced axis with readable millisecond labels
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{10 ** v:,.0f}'))
ax.set_ylabel('Execution Time (s, log scale)')
plt.tight_layout()
ps.save_fig(fig, '03_time_violin_compiler')
plt.show()

> **Takeaway:** the three compilers occupy clearly separated time bands — AOT fastest, JIT in the middle, interpreted slowest — with the Kruskal-Wallis / Mann-Whitney tests below confirming the separation is significant.

In [ ]:
def rank_biserial(x, y):
    u, _ = stats.mannwhitneyu(x, y, alternative='two-sided')
    return 1 - (2 * u) / (len(x) * len(y))

groups = {p: df[df['compiler'] == p][COL_TIME].values for p in COMPILER_ORDER}
kw_stat, kw_p = stats.kruskal(*groups.values())
n_pairs = len(COMPILER_ORDER) * (len(COMPILER_ORDER) - 1) // 2

print(f"Kruskal-Wallis (Execution Time, s): H={kw_stat:.3f}, p={kw_p:.4f}")
print("SIGNIFICANT" if kw_p < ALPHA else "Not significant")

if kw_p < ALPHA:
    print(f"\nPost-hoc (Bonferroni α={ALPHA/n_pairs:.4f}):")
    for p1, p2 in combinations(COMPILER_ORDER, 2):
        u, p = stats.mannwhitneyu(groups[p1], groups[p2], alternative='two-sided')
        p_adj = min(p * n_pairs, 1.0)
        r = rank_biserial(groups[p1], groups[p2])
        sig = "✓" if p_adj < ALPHA else "✗"
        print(f"  {sig} {p1} vs {p2}: p_adj={p_adj:.4f}, r={r:.3f}")

## 4. Benchmark-Level Time Heatmap

Mean execution time (s) for each language × benchmark cell (the per-cell means stored in
`results_clean_runs.csv`). Reveals which benchmarks are the slowest and which languages suffer
most on specific workloads.

In [ ]:
pivot_time = df_mean.pivot(index='language', columns='benchmark', values=COL_TIME)
lang_sort = lang_means(COL_TIME).sort_values().index
pivot_time = pivot_time.loc[lang_sort]

fig, ax = plt.subplots(figsize=(13, 9))
sns.heatmap(pivot_time, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax,
            linewidths=0.3, cbar_kws={'label': 'Time (s)'})
ax.set_xlabel('')
ax.set_ylabel('')
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
plt.tight_layout()
ps.save_fig(fig, '03_time_heatmap_benchmark')
plt.show()

> **Takeaway:** regex-redux and k-nucleotide are the slowest workloads, and the interpreted languages pay the largest time penalty on them.